# Video Food Classifier -- ViT with Temporal Pooling

This notebook walks through the full pipeline for classifying food videos as
**healthy** or **unhealthy** using the modular `vit_video` package.

Steps covered:
1. Setup (clone repo, install deps)
2. Environment check
3. Data download & frame extraction
4. Dataset inspection
5. Training
6. Evaluation on test split
7. Model validation (leakage audit, k-fold CV, external test)
8. Inference on a video file
9. Export to mobile formats
10. Upload to Hugging Face Hub

## 1. Setup -- Clone Repo & Install Dependencies

In [ ]:
import os

# Clone the repository (skip if already cloned)
REPO_URL = "https://github.com/whispr-messenger/moderation-service.git"
REPO_DIR = "/content/moderation-service"
BRANCH = "WHISPR-668"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    print(f"Repository already cloned at {REPO_DIR}")

# Set working directory to the vit_video package
VIT_DIR = os.path.join(REPO_DIR, "src", "vit_video")
os.chdir(VIT_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt
!pip install -q icrawler

# Restart runtime so freshly-installed C-extensions (numpy, cv2, etc.) load cleanly.
# After restart, Colab will re-run from the next cell automatically if you use "Run all".
import os
if "COLAB_RELEASE_TAG" in os.environ:
    print("✔ Packages installed. Restarting runtime …")
    os.kill(os.getpid(), 9)

## 2. Environment Check

In [ ]:
import sys, importlib
from pathlib import Path

print(f"Python {sys.version}")

for pkg in ["torch", "torchvision", "timm", "cv2", "sklearn", "icrawler"]:
    try:
        m = importlib.import_module(pkg)
        ver = getattr(m, "__version__", "ok")
        print(f"  {pkg:16s} {ver}")
    except ImportError:
        print(f"  {pkg:16s} NOT INSTALLED")

import torch
print(f"\nCUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Add src/ to sys.path so vit_video package imports work
import sys
from pathlib import Path

SRC_DIR = str(Path(VIT_DIR).parent)
if SRC_DIR not in sys.path:
    sys.path.insert(0, SRC_DIR)

from vit_video.utils.hardware import get_device, print_device_info

device = get_device()
print_device_info(hint_cuda=True)

## 3. Data Download & Frame Extraction

Downloads food images from the web (Bing image search) for each category and
organises them as pseudo-video frames. Each search keyword becomes one "video"
whose frames are the downloaded images.

In [ ]:
from vit_video.paths import DEFAULT_DATASET_DIR, DEFAULT_FRAMES_DIR
from vit_video.data.splits import frames_directory_has_images

DATASET_DIR = DEFAULT_DATASET_DIR   # vit_video/food_data
FRAMES_DIR  = DEFAULT_FRAMES_DIR    # vit_video/food_data/frames

FOOD_CATEGORIES = {
    "healthy": [
        "banana fruit close up",
        "fresh apple fruit plate",
        "broccoli vegetable cooking",
        "green salad bowl meal",
        "healthy oatmeal breakfast bowl",
        "grilled salmon vegetables plate",
        "quinoa bowl healthy meal",
        "greek yogurt berries breakfast",
        "green smoothie drink healthy",
        "avocado toast breakfast",
        "steamed vegetables plate",
        "fresh orange citrus fruit",
        "vegetable stir fry healthy cooking",
        "tofu vegetable bowl vegan",
        "mediterranean hummus plate",
        "brown rice bowl vegetables",
        "fresh mixed berries bowl",
        "kale spinach salad",
        "baked sweet potato meal",
        "lentil soup healthy",
        "whole grain toast breakfast",
        "cucumber tomato salad fresh",
        "edamame beans healthy snack",
        "grilled chicken breast vegetables",
        "sushi roll fresh fish",
        "chickpea buddha bowl",
        "overnight oats fruit jar",
        "roasted vegetables oven tray",
        "poke bowl fresh tuna",
        "fruit platter party healthy",
    ],
    "unhealthy": [
        "cheeseburger close up eating",
        "pepperoni pizza slice cheese",
        "french fries fast food",
        "glazed donut dessert",
        "fried chicken bucket crispy",
        "hot dog street food",
        "bacon cheeseburger fast food",
        "double cheeseburger fast food",
        "deep fried chicken wings",
        "fried onion rings basket",
        "mozzarella sticks fried",
        "deep fried corn dog",
        "fried fish and chips",
        "milkshake dessert drink",
        "ice cream sundae chocolate",
        "chocolate chip cookies baking",
        "chocolate cake slice frosting",
        "churros fried sugar dessert",
        "glazed donuts box dozen",
        "cotton candy carnival",
        "cinnamon roll frosting",
        "brownie chocolate fudge",
        "cupcake frosting sprinkles",
        "cheesecake slice strawberry",
        "soda cola pouring glass",
        "nachos cheese jalapeno loaded",
        "loaded fries cheese bacon",
        "candy sweets unwrapping",
        "mac and cheese creamy bowl",
        "instant ramen noodles bowl",
        "potato chips bag eating",
    ],
}

print(f"Categories: {list(FOOD_CATEGORIES.keys())}")
print(f"Total keywords: {sum(len(v) for v in FOOD_CATEGORIES.values())}")

In [ ]:
import re
import shutil
import logging
from icrawler.builtin import BingImageCrawler

IMAGES_PER_KEYWORD = 15


def download_food_images(frames_dir, categories, images_per_keyword=15):
    """Download food images from Bing and name them as pseudo-video frames.

    Each keyword produces files like  keyword_slug_frame_0000.jpg  so the
    VideoDataset groups them as one "video" per keyword.
    """
    for category, keywords in categories.items():
        cat_dir = frames_dir / category
        cat_dir.mkdir(parents=True, exist_ok=True)

        for keyword in keywords:
            slug = re.sub(r"[^a-z0-9]+", "_", keyword.lower()).strip("_")

            # Skip keywords that already have enough frames
            existing = list(cat_dir.glob(f"{slug}_frame_*.jpg"))
            if len(existing) >= images_per_keyword:
                continue

            tmp_dir = frames_dir / "_tmp_download"
            if tmp_dir.exists():
                shutil.rmtree(tmp_dir)
            tmp_dir.mkdir()

            try:
                crawler = BingImageCrawler(
                    storage={"root_dir": str(tmp_dir)},
                    log_level=logging.WARNING,
                )
                crawler.crawl(
                    keyword=keyword + " food",
                    max_num=images_per_keyword,
                )

                # Rename downloaded images into frame naming convention
                idx = 0
                for img_file in sorted(tmp_dir.iterdir()):
                    if img_file.suffix.lower() in (".jpg", ".jpeg", ".png", ".webp"):
                        dest = cat_dir / f"{slug}_frame_{idx:04d}.jpg"
                        shutil.move(str(img_file), str(dest))
                        idx += 1
            except Exception as e:
                print(f"  [WARN] {category}/{keyword}: {e}")
            finally:
                if tmp_dir.exists():
                    shutil.rmtree(tmp_dir)

        n_frames = len(list(cat_dir.glob("*.jpg")))
        print(f"  {category}: {n_frames} images")


if not frames_directory_has_images(FRAMES_DIR):
    print("Downloading food images from the web...")
    download_food_images(FRAMES_DIR, FOOD_CATEGORIES, IMAGES_PER_KEYWORD)
    print("\nDownload complete.")
else:
    print(f"Frames already exist in {FRAMES_DIR} -- skipping download.")

## 4. Dataset Inspection

In [ ]:
from vit_video.data.splits import discover_videos_by_class, count_frame_images

videos_by_class = discover_videos_by_class(FRAMES_DIR)
total_frames = count_frame_images(FRAMES_DIR)

print(f"Total frames on disk: {total_frames}\n")
for cls, stems in sorted(videos_by_class.items()):
    print(f"  {cls}: {len(stems)} unique videos")

In [ ]:
# Visualise sample frames
%matplotlib inline
import matplotlib.pyplot as plt
from vit_video.data.dataset import VideoDataset

preview_ds = VideoDataset(
    root=FRAMES_DIR,
    frames_per_video=8,
    img_size=224,
    augment=False,
)
print(f"Dataset size: {len(preview_ds)} videos, classes: {preview_ds.classes}")

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
video_tensor, label = preview_ds[0]
for i, ax in enumerate(axes.flat):
    frame = video_tensor[i].permute(1, 2, 0).numpy()
    frame = frame * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]  # un-normalise
    frame = frame.clip(0, 1)
    ax.imshow(frame)
    ax.set_title(f"Frame {i}")
    ax.axis("off")
fig.suptitle(f"Sample video -- class: {preview_ds.classes[label]}", fontsize=14)
plt.tight_layout()
plt.show()

## 5. Training

Uses `train.py` via its Python API. Key defaults:
- Backbone: ViT-B/16 on GPU, MobileViT-XXS on CPU
- 25 epochs, batch 8, lr 3e-5, dropout 0.4
- AdamW + warmup (3 epochs) + cosine annealing
- Label smoothing 0.1, gradient clipping 1.0
- Class weighting enabled by default

In [ ]:
from vit_video.data.splits import ensure_split_manifest

# Create or reuse a video-level split manifest
manifest_path = ensure_split_manifest(
    frames_root=FRAMES_DIR,
    manifest_path=DATASET_DIR / "video_split_manifest.json",
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15,
    seed=42,
)
print(f"Split manifest: {manifest_path}")

In [ ]:
import argparse
from vit_video.train import main as train_main
from vit_video.paths import PACKAGE_ROOT

# Define the output model path
MODEL_PATH = PACKAGE_ROOT / "models" / "best_food_classifier.pth"
MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)

train_args = argparse.Namespace(
    dataset_dir=str(FRAMES_DIR),
    epochs=10,
    batch_size=8,
    lr=3e-5,
    weight_decay=1e-3,
    max_grad_norm=1.0,
    dropout=0.4,
    num_frames=8,
    img_size=224,
    disable_augmentation=False,
    class_weighting=True,
    min_samples_per_class=50,
    temporal_pool="avg",
    norm_mean="0.485,0.456,0.406",
    norm_std="0.229,0.224,0.225",
    hparam_search_epochs=0,
    lr_candidates="5e-6,1e-5,3e-5,5e-5,1e-4",
    num_workers=2,
    patience=7,
    min_delta=5e-5,
    backbone="auto",
    output_model=str(MODEL_PATH),
    resume_from="",
    split_manifest=str(manifest_path),
    no_auto_split_manifest=True,
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15,
    split_seed=42,
)

train_main(train_args)

# train_main does not return the path; use the configured output path
model_path = str(MODEL_PATH)
print(f"\nSaved model: {model_path}")

### Plot training history

In [ ]:
import json
import matplotlib.pyplot as plt

history_file = Path(model_path).with_name(
    Path(model_path).stem + "_history.json"
)
if history_file.exists():
    history = json.loads(history_file.read_text())
else:
    print(f"History file not found: {history_file}")
    history = None

if history:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(history["train_loss"], label="Train", marker="o", markersize=4)
    ax1.plot(history["val_loss"], label="Val", marker="s", markersize=4)
    ax1.set_xlabel("Epoch")
    ax1.set_ylabel("Loss")
    ax1.set_title("Loss")
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    ax2.plot(history["train_acc"], label="Train", marker="o", markersize=4)
    ax2.plot(history["val_acc"], label="Val", marker="s", markersize=4)
    ax2.set_xlabel("Epoch")
    ax2.set_ylabel("Accuracy")
    ax2.set_title("Accuracy")
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## 6. Evaluation on Test Split

In [ ]:
from vit_video.test import build_test_loader, evaluate, print_results, save_results
from vit_video.utils.model_utils import load_model_from_checkpoint

model = load_model_from_checkpoint(model_path, num_classes=2, device=device)

test_loader, classes, _test_indices = build_test_loader(
    dataset_root=str(FRAMES_DIR),
    batch_size=4,
    num_frames=8,
    num_workers=2,
    split_manifest=str(manifest_path),
)

results = evaluate(model, test_loader, device, classes)
print_results(results)
save_results(results, PACKAGE_ROOT / "results")

In [ ]:
# Confusion matrix
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

cm = np.array(results["confusion_matrix"])
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=classes, yticklabels=classes, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
ax.set_title("Confusion Matrix")
plt.tight_layout()
plt.show()

print(f"\nAccuracy:  {results['accuracy']:.4f}")
print(f"Precision: {results['precision_macro']:.4f}")
print(f"Recall:    {results['recall_macro']:.4f}")
print(f"F1:        {results['f1_macro']:.4f}")

## 7. Model Validation

Runs data-leakage audit, k-fold cross-validation, and external-data testing.

In [ ]:
# Run the full validation suite via CLI
import subprocess, sys

validate_script = str(PACKAGE_ROOT / "validate_model.py")

cmd = [
    sys.executable, validate_script,
    "--model", str(model_path),
    "--dataset-dir", str(FRAMES_DIR),
    "--n-folds", "3",        # fewer folds for speed in notebook
    "--epochs", "5",         # shorter CV epochs
    "--skip-external",       # remove this flag to also test on fresh YouTube data
]
subprocess.run(cmd, check=False)

## 8. Inference on a Single Image

Since we used web images instead of videos, we test inference on a
downloaded image file.

In [ ]:
from PIL import Image
from vit_video.utils.data_utils import build_transform

transform = build_transform(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225],
)

# Grab any downloaded image to test inference
sample_images = sorted(FRAMES_DIR.glob("healthy/*.jpg"))[:1]

if sample_images:
    img_path = sample_images[0]
    img = Image.open(img_path).convert("RGB").resize((224, 224))

    # Duplicate the image into 8 "frames" to match model input shape
    import time
    video_tensor = torch.stack([transform(img)] * 8).unsqueeze(0).to(device)

    model.eval()
    with torch.no_grad():
        start = time.perf_counter()
        outputs = model(video_tensor)
        elapsed_ms = (time.perf_counter() - start) * 1000.0
        probs = torch.softmax(outputs, dim=1)
        pred_idx = torch.argmax(probs, dim=1).item()
        confidence = probs[0][pred_idx].item()

    class_names = ["healthy", "unhealthy"]
    print(f"Image:      {img_path.name}")
    print(f"Prediction: {class_names[pred_idx]}")
    print(f"Confidence: {confidence:.2%}")
    print(f"Latency:    {elapsed_ms:.0f} ms")

    plt.imshow(img)
    plt.title(f"{class_names[pred_idx]} ({confidence:.1%})")
    plt.axis("off")
    plt.show()
else:
    print("No sample images found -- run data download first (step 3).")

## 9. Export to Mobile Formats

Exports TorchScript and ONNX by default. Add `"coreml"` or `"tflite"` to the
formats list if the optional dependencies are installed.

In [ ]:
import subprocess, sys

export_script = str(PACKAGE_ROOT / "export_mobile.py")
export_dir = PACKAGE_ROOT / "exported_models"
results_file = PACKAGE_ROOT / "results" / "test_results.json"

export_cmd = [
    sys.executable, export_script,
    "--model", str(model_path),
    "--output-dir", str(export_dir),
    "--format", "torchscript", "onnx",
    "--eval-results", str(results_file),
]
subprocess.run(export_cmd, check=False)

print("\nExported files:")
for f in sorted(export_dir.glob("*")):
    size_mb = f.stat().st_size / 1024 / 1024 if f.is_file() else 0
    print(f"  {f.name:40s} {size_mb:>7.1f} MB" if f.is_file() else f"  {f.name}/")

## 10. Upload to Hugging Face Hub (Optional)

Requires `huggingface-hub` and a valid token (`huggingface-cli login`).

In [ ]:
import subprocess, sys

upload_script = str(PACKAGE_ROOT / "upload_hf.py")

subprocess.run([
    sys.executable, upload_script,
    "--repo-id", "maia2000/food-classifier",
    "--export-dir", str(export_dir),
], check=False)

## Model Summary

In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable   = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Model Summary")
print(f"  Architecture:       MobileViTModel")
print(f"  Backbone:           {getattr(model, 'model_name', 'auto-detected')}")
print(f"  Temporal pooling:   {getattr(model, 'temporal_pool', 'avg')}")
print(f"  Total parameters:   {total_params:,}")
print(f"  Trainable params:   {trainable:,}")
print(f"  Input shape:        (B, 8, 3, 224, 224)")
print(f"  Output:             (B, 2)  [healthy, unhealthy]")
print(f"  Device:             {device}")